In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
df = pd.read_csv('dataset.csv', index_col=0)

In [3]:
df.shape    # (antall rader, antall kolonner)
df.head()   # de 5 første radene
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  str    
 1   artists           113999 non-null  str    
 2   album_name        113999 non-null  str    
 3   track_name        113999 non-null  str    
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           114000 non-nu

In [4]:
df.isna().sum()      # antall manglende per kolonne
df = df.dropna()     # fjern radene som mangler noe

In [5]:
df['track_genre'].nunique()        # antall sjangre
df['track_genre'].value_counts()   # sanger per sjanger

track_genre
acoustic       1000
afrobeat       1000
alt-rock       1000
alternative    1000
ambient        1000
               ... 
trance         1000
trip-hop       1000
turkish        1000
world-music    1000
k-pop           999
Name: count, Length: 114, dtype: int64

In [6]:
df['track_id'].duplicated().sum()  # sanger som finnes flere ganger

np.int64(24259)

## Oppsett
Funksjonen `kjør_par` gjør hele analysen for house (y=1) mot en annen sjanger (y=0).

In [ ]:
import warnings

egenskaper = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo']

In [ ]:
def kjør_par(sjanger0):
    """Klassifiserer house (y=1) mot sjanger0 (y=0) og returnerer resultatene."""
    fil = f'house_{sjanger0}'
    res = {}

    # 1. Velg sjangrene, fjern sanger som finnes under begge, og tell
    par = df[df['track_genre'].isin(['house', sjanger0])]
    par = par.drop_duplicates(subset='track_id', keep=False)
    antall = par['track_genre'].value_counts()
    print("Antall sanger per sjanger:")
    print(antall)
    andel_storst = antall.max() / antall.sum()
    res['antall'] = antall.to_dict()
    res['andel_storst'] = andel_storst
    if antall.nunique() > 1:
        print(f"OBS: ulikt antall sanger. Andel i største klasse: {andel_storst:.4f} "
              f"(tilfeldig gjetting er ikke nøyaktig 0.5)")

    # 2. X og y
    X = par[egenskaper]
    y = (par['track_genre'] == 'house').astype(int)   # house = 1

    # 3. Deling 70/15/15
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=10)
    X_test, X_val, y_test, y_val = train_test_split(
        X_test, y_test, test_size=0.5, stratify=y_test, random_state=10)
    print("Størrelse trening/validering/test:", len(X_train), len(X_val), len(X_test))

    # 4. Standardisering tilpasset kun på treningssettet
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # 5. Boksplott av de 9 lydegenskapene
    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    fig.suptitle(f'Lydegenskaper for house og {sjanger0}')
    for ax, e in zip(axes.flat, egenskaper):
        sns.boxplot(data=par, x='track_genre', y=e, ax=ax)
        ax.set_title(e)
        ax.set_xlabel('')
    plt.tight_layout()
    plt.show()

    # 6. Logistisk regresjon (statsmodels): full og redusert modell
    cutoff = 0.5
    df_tren = pd.DataFrame(X_train, columns=egenskaper)
    df_tren['y'] = y_train.values
    df_val = pd.DataFrame(X_val, columns=egenskaper)
    df_val['y'] = y_val.values
    df_test = pd.DataFrame(X_test, columns=egenskaper)

    with warnings.catch_warnings(record=True) as advarsler:
        warnings.simplefilter('always')
        res_full = smf.logit('y ~ ' + ' + '.join(egenskaper), data=df_tren).fit(disp=0)
        p_verdier = res_full.pvalues.drop('Intercept')
        beholdt = list(p_verdier[p_verdier < 0.05].index)
        fjernet = [e for e in egenskaper if e not in beholdt]
        res_red = smf.logit('y ~ ' + ' + '.join(beholdt), data=df_tren).fit(disp=0)
    res['advarsler'] = sorted({str(a.message) for a in advarsler})
    for a in res['advarsler']:
        print("ADVARSEL fra statsmodels:", a)
    print(res_full.summary())
    print("Beholdt:", beholdt)
    print("Fjernet:", fjernet)
    res['fjernet'] = fjernet

    feil_val = {}
    for navn, m in [('Full', res_full), ('Redusert', res_red)]:
        pred = np.where(m.predict(exog=df_val) > cutoff, 1, 0)
        feil_val[navn] = 1 - accuracy_score(df_val['y'], pred)
        print(navn, "- feilrate validering:", round(feil_val[navn], 4))
    # ved lik feilrate velger vi den enkleste modellen (den reduserte)
    valgt_navn = 'Redusert' if feil_val['Redusert'] <= feil_val['Full'] else 'Full'
    valgt = res_red if valgt_navn == 'Redusert' else res_full
    print("Valgt modell:", valgt_navn)
    res['valgt_modell'] = valgt_navn
    res['val_feil_lr'] = feil_val[valgt_navn]
    print(valgt.summary())
    res['koeff'] = valgt.params
    res['exp_koeff'] = np.exp(valgt.params)
    print(pd.DataFrame({'beta': valgt.params, 'exp(beta)': np.exp(valgt.params)}))

    # feilrate for ulike cutoff (bare til rapporten; cutoff = 0.5 brukes videre)
    val_prob = valgt.predict(exog=df_val)
    for c in np.arange(0.1, 1.0, 0.1):
        pred = np.where(val_prob > c, 1, 0)
        print("cutoff:", round(c, 1), "feilrate:", round(1 - accuracy_score(df_val['y'], pred), 4))

    # koeffisientfigur for den valgte modellen
    koef = valgt.params.drop('Intercept').sort_values()
    koef.plot(kind='barh')
    plt.xlabel(f'Koeffisient (positiv = house, negativ = {sjanger0})')
    plt.title(f'Koeffisienter i logistisk regresjon, house mot {sjanger0} (standardiserte variabler)')
    plt.show()

    # 7. kNN: feilrate på valideringssettet for k = 1, 3, ..., 21
    knaboer = np.arange(1, 22, 2)
    val_feilrate = []
    for kk in knaboer:
        knn = KNeighborsClassifier(n_neighbors=kk)
        knn.fit(X_train, y_train)
        val_feilrate.append(1 - knn.score(X_val, y_val))
    plt.plot(knaboer, val_feilrate, marker='o')
    plt.xlabel('Antall naboer k')
    plt.ylabel('Feilrate (validering)')
    plt.title(f'Feilrate på valideringssettet for ulike k, house mot {sjanger0}')
    plt.show()
    print(pd.DataFrame({'k': knaboer, 'feilrate': val_feilrate}))
    k = knaboer[np.argmin(val_feilrate)]   # k med lavest feilrate på valideringssettet
    print("Valgt k:", k, "- feilrate validering:", round(min(val_feilrate), 4))
    res['k'] = k
    res['val_feil_knn'] = min(val_feilrate)

    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred_knn = knn.predict(X_test)
    y_pred_lr = np.where(valgt.predict(exog=df_test) > cutoff, 1, 0)

    # 8 og 9. Evaluering på testsettet og hypotesetest mot tilfeldig gjetting
    n = len(y_test)
    res['konf'] = {}
    for navn, y_pred in [('LR', y_pred_lr), ('KNN', y_pred_knn)]:
        konf = pd.DataFrame(confusion_matrix(y_test, y_pred),
                            index=[f'true:{sjanger0}', 'true:house'],
                            columns=[f'pred:{sjanger0}', 'pred:house'])
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        p_hat = accuracy_score(y_test, y_pred)
        Z = (p_hat - 0.5) / np.sqrt(0.5 * 0.5 / n)
        res[navn] = {'Feilrate': 1 - p_hat, 'Sensitivitet': tp / (tp + fn),
                     'Spesifisitet': tn / (tn + fp), 'p_hat': p_hat, 'Z': Z}
        res['konf'][navn] = konf
        print(f"\n{navn} på testsettet:")
        print(konf)
        print(f"p_hat = {p_hat:.4f}, Z = {Z:.2f}, forkast H0 (Z > 1.645): {Z > 1.645}")
    return res

## Runde 1: house mot trance

In [ ]:
res_trance = kjør_par('trance')

## Runde 2: house mot classical

In [ ]:
res_classical = kjør_par('classical')

## Samlet resultat

In [ ]:
samlet = pd.DataFrame({
    'LR house/trance': res_trance['LR'], 'KNN house/trance': res_trance['KNN'],
    'LR house/classical': res_classical['LR'], 'KNN house/classical': res_classical['KNN']
}).loc[['Feilrate', 'Sensitivitet', 'Spesifisitet', 'p_hat', 'Z']]
samlet.round(4).to_csv('resultater_klassifikasjon.csv')
samlet.round(4)